# Polars: Complete Practical Practice Guide

This notebook is designed as a reusable **Polars syntax bank + hands-on data engineering reference**.

It uses the publicly available **NYC Taxi & Limousine Commission (TLC) Yellow Taxi Trip Records** dataset so that the examples operate on realistic data stored in Parquet format.

## Official documentation

### Polars
- [Polars User Guide](https://docs.pola.rs/user-guide/)
- [Getting Started](https://docs.pola.rs/user-guide/getting-started/)
- [Python API Reference](https://docs.pola.rs/api/python/stable/reference/)
- [Expressions](https://docs.pola.rs/user-guide/expressions/)
- [Lazy API](https://docs.pola.rs/user-guide/lazy/)
- [Joins](https://docs.pola.rs/user-guide/transformations/joins/)
- [Missing Data](https://docs.pola.rs/user-guide/expressions/missing-data/)
- [SQL Interface](https://docs.pola.rs/user-guide/sql/intro/)

### Dataset documentation
- [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
- [Yellow Taxi Trip Records Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)
- [TLC Trip Record User Guide](https://www.nyc.gov/assets/tlc/downloads/pdf/trip_record_user_guide.pdf)
- [Taxi Zone Lookup Table](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv)

## Dataset used

This notebook uses:
- **Yellow Taxi Trip Records — January 2024**
- **NYC Taxi Zone Lookup Table**

Yellow taxi records include pickup/drop-off timestamps, trip distance, passenger count, pickup and drop-off zone IDs, payment type, fares, tips, tolls, and total amounts.

> This notebook downloads public data only. No private or proprietary data is required.

## 1. Installation and imports

Install Polars once in your environment if needed.

In [6]:
# Run only if needed:
#%pip install -U polars pyarrow

In [8]:
import polars as pl
from pathlib import Path
from urllib.request import urlretrieve

print("Polars version:", pl.__version__)

Polars version: 1.44.1


## 2. Download the public practice data

The dataset is stored locally in a `data/` folder. For a GitHub repo, add `data/` to `.gitignore` so the large public data file is not committed.

In [11]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TRIP_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
ZONE_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

TRIP_FILE = DATA_DIR / "yellow_tripdata_2024-01.parquet"
ZONE_FILE = DATA_DIR / "taxi_zone_lookup.csv"

if not TRIP_FILE.exists():
    print("Downloading Yellow Taxi data...")
    urlretrieve(TRIP_URL, TRIP_FILE)

if not ZONE_FILE.exists():
    print("Downloading taxi zone lookup...")
    urlretrieve(ZONE_URL, ZONE_FILE)

print(TRIP_FILE)
print(ZONE_FILE)

data\yellow_tripdata_2024-01.parquet
data\taxi_zone_lookup.csv


# Part I — Core Polars DataFrame Skills

## 3. Read data eagerly

`pl.read_parquet()` immediately reads a file into memory and returns a `DataFrame`.

In [14]:
df = pl.read_parquet(TRIP_FILE)
df.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


## 4. Inspect a DataFrame

In [16]:
print("Shape:", df.shape)
print("Rows:", df.height)
print("Columns:", df.width)
df.schema

Shape: (2964624, 19)
Rows: 2964624
Columns: 19


Schema([('VendorID', Int32),
        ('tpep_pickup_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('RatecodeID', Int64),
        ('store_and_fwd_flag', String),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('Airport_fee', Float64)])

In [18]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [20]:
df.head(5)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


In [22]:
df.tail(5)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-31 23:45:59,2024-01-31 23:54:36,null,3.18,null,null,107,263,0,15.77,0.0,0.5,2.0,0.0,1.0,21.77,null,null
1,2024-01-31 23:13:07,2024-01-31 23:27:52,null,4.0,null,null,114,236,0,18.4,1.0,0.5,2.34,0.0,1.0,25.74,null,null
2,2024-01-31 23:19:00,2024-01-31 23:38:00,null,3.33,null,null,211,25,0,19.97,0.0,0.5,0.0,0.0,1.0,23.97,null,null
2,2024-01-31 23:07:23,2024-01-31 23:25:14,null,3.06,null,null,107,13,0,23.88,0.0,0.5,5.58,0.0,1.0,33.46,null,null
1,2024-01-31 23:58:25,2024-02-01 00:13:30,null,8.1,null,null,138,75,0,32.4,7.75,0.5,7.29,6.94,1.0,55.88,null,null


In [24]:
df.sample(5, seed=42)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-27 16:42:57,2024-01-27 16:55:37,1,1.02,1,"""N""",79,234,1,12.1,0.0,0.5,2.0,0.0,1.0,18.1,2.5,0.0
2,2024-01-12 06:27:09,2024-01-12 06:33:39,1,0.91,1,"""N""",100,161,1,7.9,0.0,0.5,1.0,0.0,1.0,12.9,2.5,0.0
2,2024-01-20 23:19:20,2024-01-20 23:38:09,null,2.93,null,null,229,79,0,20.41,0.0,0.5,0.0,0.0,1.0,24.41,null,null
2,2024-01-24 12:35:30,2024-01-24 12:57:17,1,7.53,1,"""N""",140,243,2,32.4,0.0,0.5,0.0,0.0,1.0,36.4,2.5,0.0
2,2024-01-26 23:35:31,2024-01-26 23:43:18,1,1.47,1,"""N""",237,161,1,10.0,1.0,0.5,2.0,0.0,1.0,17.0,2.5,0.0


In [26]:
df.describe()

statistic,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
str,f64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",2.964624e6,"""2964624""","""2964624""",2.824462e6,2.964624e6,2.824462e6,"""2824462""",2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.964624e6,2.824462e6,2.824462e6
"""null_count""",0.0,"""0""","""0""",140162.0,0.0,140162.0,"""140162""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,140162.0,140162.0
"""mean""",1.754204,"""2024-01-17 00:46:36.431093""","""2024-01-17 01:02:13.208130""",1.339281,3.652169,2.069359,null,166.017884,165.116712,1.161271,18.175062,1.451598,0.483382,3.33587,0.527021,0.975632,26.801505,2.256122,0.141161
"""std""",0.43259,null,null,0.850282,225.462572,9.823219,null,63.623914,69.31535,0.580869,18.949548,1.804102,0.11776,3.896551,2.12831,0.218364,23.385577,0.823275,0.487624
"""min""",1.0,"""2002-12-31 22:59:39""","""2002-12-31 23:05:41""",0.0,0.0,1.0,"""N""",1.0,1.0,0.0,-899.0,-7.5,-0.5,-80.0,-80.0,-1.0,-900.0,-2.5,-1.75
"""25%""",2.0,"""2024-01-09 15:59:20""","""2024-01-09 16:16:23""",1.0,1.0,1.0,null,132.0,114.0,1.0,8.6,0.0,0.5,1.0,0.0,1.0,15.38,2.5,0.0
"""50%""",2.0,"""2024-01-17 10:45:38""","""2024-01-17 11:03:52""",1.0,1.68,1.0,null,162.0,162.0,1.0,12.8,1.0,0.5,2.7,0.0,1.0,20.1,2.5,0.0
"""75%""",2.0,"""2024-01-24 18:23:52""","""2024-01-24 18:40:29""",1.0,3.11,1.0,null,234.0,234.0,1.0,20.5,2.5,0.5,4.12,0.0,1.0,28.56,2.5,0.0
"""max""",6.0,"""2024-02-01 00:01:15""","""2024-02-02 13:56:52""",9.0,312722.3,99.0,"""Y""",265.0,265.0,4.0,5000.0,14.25,4.0,428.0,115.92,1.0,5000.0,2.5,1.75


## 5. Select columns

Polars relies heavily on **expressions**. `pl.col("column")` is one of the most important pieces of syntax.

In [28]:
df.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
).head()

tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,fare_amount,tip_amount,total_amount
datetime[ns],datetime[ns],i64,f64,f64,f64,f64
2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,17.7,0.0,22.7
2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,10.0,3.75,18.75
2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,23.3,3.0,31.3
2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,10.0,2.0,17.0
2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,7.9,3.2,16.1


In [30]:
df.select(
    pl.col("trip_distance"),
    pl.col("fare_amount"),
    pl.col("tip_amount"),
).head()

trip_distance,fare_amount,tip_amount
f64,f64,f64
1.72,17.7,0.0
1.8,10.0,3.75
4.7,23.3,3.0
1.4,10.0,2.0
0.8,7.9,3.2


## 6. Rename columns

In [32]:
df.rename({
    "PULocationID": "pickup_location_id",
    "DOLocationID": "dropoff_location_id",
}).select(
    "pickup_location_id",
    "dropoff_location_id",
).head()

pickup_location_id,dropoff_location_id
i32,i32
186,79
140,236
236,79
79,211
211,148


## 7. `with_columns`: create or transform columns

Polars encourages expression-based transformations instead of row-by-row operations.

In [37]:
df_enriched = df.with_columns(
    (pl.col("tip_amount") / pl.col("fare_amount")).alias("tip_to_fare_ratio"),
    (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).alias("trip_duration"),
)

df_enriched.select(
    "fare_amount",
    "tip_amount",
    "tip_to_fare_ratio",
    "tpep_dropoff_datetime",
    "tpep_pickup_datetime",
    "trip_duration",
).head()

fare_amount,tip_amount,tip_to_fare_ratio,tpep_dropoff_datetime,tpep_pickup_datetime,trip_duration
f64,f64,f64,datetime[ns],datetime[ns],duration[ns]
17.7,0.0,0.0,2024-01-01 01:17:43,2024-01-01 00:57:55,19m 48s
10.0,3.75,0.375,2024-01-01 00:09:36,2024-01-01 00:03:00,6m 36s
23.3,3.0,0.128755,2024-01-01 00:35:01,2024-01-01 00:17:06,17m 55s
10.0,2.0,0.2,2024-01-01 00:44:56,2024-01-01 00:36:38,8m 18s
7.9,3.2,0.405063,2024-01-01 00:52:57,2024-01-01 00:46:51,6m 6s


## 8. Literals and aliases

In [42]:
# Create a column called dataset_type where the value is "yellow_taxi".
df.with_columns(
    pl.lit("yellow_taxi").alias("dataset_type")
).select(
    "dataset_type",
    "trip_distance",
).head()

dataset_type,trip_distance
str,f64
"""yellow_taxi""",1.72
"""yellow_taxi""",1.8
"""yellow_taxi""",4.7
"""yellow_taxi""",1.4
"""yellow_taxi""",0.8


## 9. Filter rows

In [45]:
df.filter(
    pl.col("trip_distance") > 10
).select(
    "trip_distance",
    "fare_amount",
    "total_amount",
).head()

trip_distance,fare_amount,total_amount
f64,f64,f64
10.82,45.7,64.95
23.9,120.0,127.94
11.51,44.3,67.49
11.48,47.8,63.36
20.85,70.0,82.69


In [47]:
df.filter(
    (pl.col("trip_distance") >= 5) &
    (pl.col("trip_distance") <= 10) &
    (pl.col("fare_amount") > 0)
).select(
    "trip_distance",
    "fare_amount",
).head()

trip_distance,fare_amount
f64,f64
5.44,31.0
8.2,59.0
5.0,21.2
5.88,28.9
5.1,28.9


In [49]:
df.filter(
    pl.col("payment_type").is_in([1, 2])
).select(
    "payment_type",
    "fare_amount",
).head()

payment_type,fare_amount
i64,f64
2,17.7
1,10.0
1,23.3
1,10.0
1,7.9


## 10. Sort

In [52]:
df.sort(
    "total_amount",
    descending=True
).select(
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
).head(10)

trip_distance,fare_amount,tip_amount,total_amount
f64,f64,f64,f64
0.0,5000.0,0.0,5000.0
0.0,5000.0,0.0,5000.0
0.0,2500.0,0.0,2500.0
0.0,2500.0,0.0,2500.0
0.0,2500.0,0.0,2500.0
31.95,2221.3,0.0,2225.3
233.25,1616.5,0.0,1617.5
0.0,1000.0,0.0,1000.0
142.62,912.3,0.0,940.93


## 11. Unique values and duplicates

In [55]:
df.select(
    pl.col("payment_type").unique().sort()
)

payment_type
i64
0
1
2
3
4


In [57]:
df.select(
    pl.col("PULocationID").n_unique().alias("unique_pickup_zones")
)

unique_pickup_zones
u32
260


In [59]:
df_unique = df.unique()
print(df.shape)
print(df_unique.shape)

(2964624, 19)
(2964624, 19)


# Part II — Expressions and Data Cleaning

## 12. Null values

Polars distinguishes **null** values from floating-point **NaN** values.

In [65]:
df.null_count()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,140162,0,140162,140162,0,0,0,0,0,0,0,0,0,0,140162,140162


In [67]:
df.filter(
    pl.col("passenger_count").is_null()
).head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:34:19,2024-01-01 00:51:22,null,2.04,null,null,143,141,0,12.72,0.0,0.5,0.0,0.0,1.0,16.72,null,null
1,2024-01-01 00:14:31,2024-01-01 00:19:29,null,1.6,null,null,236,238,0,9.3,1.0,0.5,2.86,0.0,1.0,17.16,null,null
1,2024-01-01 00:35:11,2024-01-01 01:13:40,null,0.0,null,null,142,79,0,21.01,0.0,0.5,0.0,0.0,1.0,25.01,null,null
1,2024-01-01 00:33:37,2024-01-01 00:50:34,null,0.0,null,null,237,4,0,17.79,0.0,0.5,0.0,0.0,1.0,21.79,null,null
1,2024-01-01 00:49:04,2024-01-01 01:01:16,null,0.0,null,null,244,50,0,34.65,0.0,0.5,0.0,0.0,1.0,38.65,null,null


### See [polars.Expr.fill_null](https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.Expr.fill_null.html) documentation for other options such as forward,backward,mean

In [69]:
df.with_columns(
    pl.col("passenger_count")
      .fill_null(pl.col("passenger_count").median())
      .alias("passenger_count_filled")
).select(
    "passenger_count",
    "passenger_count_filled",
).head()

passenger_count,passenger_count_filled
i64,f64
1,1.0
1,1.0
1,1.0
1,1.0
1,1.0


## 13. Conditional logic: `when / then / otherwise`

In [73]:
df.with_columns(
    pl.when(pl.col("trip_distance") < 2)
      .then(pl.lit("short"))
      .when(pl.col("trip_distance") < 10)
      .then(pl.lit("medium"))
      .otherwise(pl.lit("long"))
      .alias("trip_category")
).select(
    "trip_distance",
    "trip_category",
).head(10)

trip_distance,trip_category
f64,str
1.72,"""short"""
1.8,"""short"""
4.7,"""medium"""
1.4,"""short"""
0.8,"""short"""
4.7,"""medium"""
10.82,"""long"""
3.0,"""medium"""
5.44,"""medium"""


## 14. Casting data types

In [75]:
df.select(
    pl.col("passenger_count"),
    pl.col("passenger_count").cast(pl.Int64, strict=False).alias("passenger_count_int"),
).head()

passenger_count,passenger_count_int
i64,i64
1,1
1,1
1,1
1,1
1,1


## 15. Column selectors

In [82]:
import polars.selectors as cs

df.select(
    cs.numeric()
).head()

VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,i64,f64,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,1,1.72,1,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,1,1.8,1,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,1,4.7,1,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,1,1.4,1,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,1,0.8,1,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


In [84]:
df.select(
    cs.float().round(2)
).head()

trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1.72,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1.8,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
4.7,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1.4,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
0.8,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


# Part III — Aggregation

## 16. Basic aggregations

In [88]:
df.select(
    pl.len().alias("row_count"),
    pl.col("trip_distance").mean().alias("avg_trip_distance"),
    pl.col("trip_distance").median().alias("median_trip_distance"),
    pl.col("fare_amount").mean().alias("avg_fare"),
    pl.col("total_amount").sum().alias("total_revenue"),
)

row_count,avg_trip_distance,median_trip_distance,avg_fare,total_revenue
u32,f64,f64,f64,f64
2964624,3.652169,1.68,18.175062,7.9456e7


## 17. Group by

In [95]:
payment_summary = (
    df.group_by("payment_type")
      .agg(
          pl.len().alias("trip_count"),
          pl.col("trip_distance").mean().alias("avg_trip_distance"),
          pl.col("fare_amount").mean().alias("avg_fare"),
          pl.col("tip_amount").mean().alias("avg_tip"),
          pl.col("total_amount").sum().alias("total_amount"),
      )
      .sort("payment_type", descending=False)
)

payment_summary

payment_type,trip_count,avg_trip_distance,avg_fare,avg_tip,total_amount
i64,u32,f64,f64,f64,f64
0,140162,11.674403,20.016194,1.545957,3.6178e6
1,2319046,3.264649,18.557432,4.169671,6.5534e7
2,439191,3.259126,17.866037,0.002296,1.0051e7
3,19597,2.159218,6.752569,0.01456,171581.04
4,46628,3.140549,1.334889,0.042125,82710.08


## 18. Multiple grouping columns

In [104]:
df.with_columns(
    pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_weekday")
).group_by(
    "pickup_weekday",
    "payment_type",
).agg(
    pl.len().alias("trip_count"),
    pl.col("total_amount").mean().alias("avg_total_amount"),
).sort(
    ["pickup_weekday", "payment_type"],
    descending=[False, True],
).head(20)

pickup_weekday,payment_type,trip_count,avg_total_amount
i8,i64,u32,f64
1,4,6697,1.7698
1,3,3134,8.764748
1,2,64732,23.584931
1,1,312701,29.972333
1,0,21013,27.80373
…,…,…,…
4,4,6301,2.015996
4,3,2674,8.335718
4,2,62626,23.351841


# Part IV — Date and Time Operations

## 19. Extract datetime components

In [108]:
df_time = df.with_columns(
    pl.col("tpep_pickup_datetime").dt.date().alias("pickup_date"),
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
    pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_weekday"),
    pl.col("tpep_pickup_datetime").dt.month().alias("pickup_month"),
)

df_time.select(
    "tpep_pickup_datetime",
    "pickup_date",
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
).head()

tpep_pickup_datetime,pickup_date,pickup_hour,pickup_weekday,pickup_month
datetime[ns],date,i8,i8,i8
2024-01-01 00:57:55,2024-01-01,0,1,1
2024-01-01 00:03:00,2024-01-01,0,1,1
2024-01-01 00:17:06,2024-01-01,0,1,1
2024-01-01 00:36:38,2024-01-01,0,1,1
2024-01-01 00:46:51,2024-01-01,0,1,1


## 20. Trip duration

In [111]:
df_time = df_time.with_columns(
    (
        pl.col("tpep_dropoff_datetime") -
        pl.col("tpep_pickup_datetime")
    ).dt.total_minutes().alias("trip_minutes")
)

df_time.select(
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_minutes",
).head()

tpep_pickup_datetime,tpep_dropoff_datetime,trip_minutes
datetime[ns],datetime[ns],i64
2024-01-01 00:57:55,2024-01-01 01:17:43,19
2024-01-01 00:03:00,2024-01-01 00:09:36,6
2024-01-01 00:17:06,2024-01-01 00:35:01,17
2024-01-01 00:36:38,2024-01-01 00:44:56,8
2024-01-01 00:46:51,2024-01-01 00:52:57,6


## 21. Aggregate by hour

In [114]:
hourly_summary = (
    df_time.group_by("pickup_hour")
           .agg(
               pl.len().alias("trip_count"),
               pl.col("trip_distance").mean().alias("avg_distance"),
               pl.col("total_amount").mean().alias("avg_total_amount"),
           )
           .sort("pickup_hour")
)

hourly_summary

pickup_hour,trip_count,avg_distance,avg_total_amount
i8,u32,f64,f64
0,79094,3.73285,27.770448
1,53627,3.127259,25.378651
2,37517,2.883984,24.084306
3,24811,3.321278,25.905998
4,16742,4.545094,30.99066
…,…,…,…
19,184032,3.11093,27.077136
20,159989,3.313008,26.72161
21,160888,3.396285,26.87756


# Part V — Strings

## 22. Load the taxi-zone lookup table

In [118]:
zones = pl.read_csv(ZONE_FILE)
zones.head()

LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


## 23. String operations

In [121]:
zones.with_columns(
    pl.col("Borough").str.to_lowercase().alias("borough_lower"),
    pl.col("Zone").str.to_uppercase().alias("zone_upper"),
    pl.col("Zone").str.len_chars().alias("zone_name_length"),
).head()

LocationID,Borough,Zone,service_zone,borough_lower,zone_upper,zone_name_length
i64,str,str,str,str,str,u32
1,"""EWR""","""Newark Airport""","""EWR""","""ewr""","""NEWARK AIRPORT""",14
2,"""Queens""","""Jamaica Bay""","""Boro Zone""","""queens""","""JAMAICA BAY""",11
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone""","""bronx""","""ALLERTON/PELHAM GARDENS""",23
4,"""Manhattan""","""Alphabet City""","""Yellow Zone""","""manhattan""","""ALPHABET CITY""",13
5,"""Staten Island""","""Arden Heights""","""Boro Zone""","""staten island""","""ARDEN HEIGHTS""",13


In [123]:
zones.filter(
    pl.col("Zone").str.contains("Airport")
)

LocationID,Borough,Zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
132,"""Queens""","""JFK Airport""","""Airports"""
138,"""Queens""","""LaGuardia Airport""","""Airports"""


# Part VI — Joins and Combining Data

## 24. Join pickup zone metadata

Polars supports inner, left, right, full, semi, anti, cross, and as-of joins.

In [127]:
pickup_zones = zones.rename({
    "LocationID": "PULocationID",
    "Borough": "pickup_borough",
    "Zone": "pickup_zone",
    "service_zone": "pickup_service_zone",
})

df_joined = df.join(
    pickup_zones,
    on="PULocationID",
    how="left",
)

df_joined.select(
    "PULocationID",
    "pickup_borough",
    "pickup_zone",
    "trip_distance",
    "total_amount",
).head()

PULocationID,pickup_borough,pickup_zone,trip_distance,total_amount
i32,str,str,f64,f64
186,"""Manhattan""","""Penn Station/Madison Sq West""",1.72,22.7
140,"""Manhattan""","""Lenox Hill East""",1.8,18.75
236,"""Manhattan""","""Upper East Side North""",4.7,31.3
79,"""Manhattan""","""East Village""",1.4,17.0
211,"""Manhattan""","""SoHo""",0.8,16.1


## 25. Join pickup and drop-off zone metadata

In [130]:
dropoff_zones = zones.rename({
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough",
    "Zone": "dropoff_zone",
    "service_zone": "dropoff_service_zone",
})

trips_with_zones = (
    df.join(
        pickup_zones,
        on="PULocationID",
        how="left",
    )
    .join(
        dropoff_zones,
        on="DOLocationID",
        how="left",
    )
)

trips_with_zones.select(
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone",
    "trip_distance",
    "total_amount",
).head()

pickup_borough,pickup_zone,dropoff_borough,dropoff_zone,trip_distance,total_amount
str,str,str,str,f64,f64
"""Manhattan""","""Penn Station/Madison Sq West""","""Manhattan""","""East Village""",1.72,22.7
"""Manhattan""","""Lenox Hill East""","""Manhattan""","""Upper East Side North""",1.8,18.75
"""Manhattan""","""Upper East Side North""","""Manhattan""","""East Village""",4.7,31.3
"""Manhattan""","""East Village""","""Manhattan""","""SoHo""",1.4,17.0
"""Manhattan""","""SoHo""","""Manhattan""","""Lower East Side""",0.8,16.1


## 26. Route-level aggregation

In [133]:
top_routes = (
    trips_with_zones
    .group_by(
        "pickup_zone",
        "dropoff_zone",
    )
    .agg(
        pl.len().alias("trip_count"),
        pl.col("trip_distance").mean().alias("avg_distance"),
        pl.col("total_amount").mean().alias("avg_total_amount"),
    )
    .sort("trip_count", descending=True)
)

top_routes.head(20)

pickup_zone,dropoff_zone,trip_count,avg_distance,avg_total_amount
str,str,u32,f64,f64
"""Upper East Side South""","""Upper East Side North""",21883,1.058019,15.508877
"""Upper East Side North""","""Upper East Side South""",19402,1.0446,15.878771
"""Upper East Side North""","""Upper East Side North""",15955,0.623932,13.050416
"""Upper East Side South""","""Upper East Side South""",14938,0.621655,13.529931
"""Midtown Center""","""Upper East Side South""",10275,1.074143,16.735615
…,…,…,…,…
"""Penn Station/Madison Sq West""","""Times Sq/Theatre District""",7085,1.057366,18.371273
"""Lincoln Square East""","""Upper West Side North""",7073,1.498542,17.198344
"""Upper West Side North""","""Upper West Side South""",7007,0.784294,13.449441


## 27. Concatenate DataFrames

In [138]:
sample_a = df.head(3)
sample_b = df.slice(3, 3)

pl.concat([sample_a, sample_b])

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,"""N""",148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0


# Part VII — Reshaping

## 28. Pivot

In [142]:
pivot_source = (
    df_time
    .group_by("pickup_hour", "payment_type")
    .agg(pl.len().alias("trip_count"))
)

pivot_table = pivot_source.pivot(
    values="trip_count",
    index="pickup_hour",
    on="payment_type",
    aggregate_function="sum",
)

pivot_table.head()

pickup_hour,4,1,2,3,0
i8,u32,u32,u32,u32,u32
5,526,12581,3433,206,2018
18,3185,170466,27662,1279,10196
10,1852,107700,24167,876,4183
16,2950,150687,30005,1292,5267
4,587,10312,2742,241,2860


## 29. Unpivot

In [145]:
small = df.select(
    "fare_amount",
    "tip_amount",
    "tolls_amount",
).head(5)

small.unpivot(
    variable_name="charge_type",
    value_name="amount",
)

charge_type,amount
str,f64
"""fare_amount""",17.7
"""fare_amount""",10.0
"""fare_amount""",23.3
"""fare_amount""",10.0
"""fare_amount""",7.9
…,…
"""tolls_amount""",0.0
"""tolls_amount""",0.0
"""tolls_amount""",0.0


# Part VIII — Window Functions

## 30. Window expressions with `.over()`

In [149]:
df.with_columns(
    pl.col("fare_amount")
      .mean()
      .over("payment_type")
      .alias("payment_type_avg_fare")
).select(
    "payment_type",
    "fare_amount",
    "payment_type_avg_fare",
).head(10)

payment_type,fare_amount,payment_type_avg_fare
i64,f64,f64
2,17.7,17.866037
1,10.0,18.557432
1,23.3,18.557432
1,10.0,18.557432
1,7.9,18.557432
1,29.6,18.557432
1,45.7,18.557432
2,25.4,17.866037
2,31.0,17.866037


In [151]:
df.with_columns(
    (
        pl.col("fare_amount") -
        pl.col("fare_amount").mean().over("payment_type")
    ).alias("fare_vs_payment_mean")
).select(
    "payment_type",
    "fare_amount",
    "fare_vs_payment_mean",
).head()

payment_type,fare_amount,fare_vs_payment_mean
i64,f64,f64
2,17.7,-0.166037
1,10.0,-8.557432
1,23.3,4.742568
1,10.0,-8.557432
1,7.9,-10.657432


# Part IX — Lazy Execution

Lazy execution is one of the most important Polars concepts for data engineering.

Instead of immediately loading and processing the entire dataset, Polars builds a **query plan** and optimizes it before execution.

Common optimizations include:
- predicate pushdown,
- projection pushdown,
- expression simplification,
- join optimization.

Use `scan_*` instead of `read_*` to begin a lazy pipeline.

## 31. `scan_parquet`

In [155]:
lazy_trips = pl.scan_parquet(TRIP_FILE)
lazy_trips

## 32. Build a lazy query

In [160]:
lazy_query = (
    pl.scan_parquet(TRIP_FILE)
    .filter(
        (pl.col("trip_distance") > 0) &
        (pl.col("fare_amount") > 0)
    )
    .with_columns(
        pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
        (
            pl.col("tip_amount") /
            pl.col("fare_amount")
        ).alias("tip_ratio"),
    )
    .group_by("pickup_hour")
    .agg(
        pl.len().alias("trip_count"),
        pl.col("trip_distance").mean().alias("avg_distance"),
        pl.col("fare_amount").mean().alias("avg_fare"),
        pl.col("tip_ratio").mean().alias("avg_tip_ratio"),
    )
    .sort("pickup_hour")
)

lazy_query

## 33. Inspect the optimized query plan

In [163]:
print(lazy_query.explain(optimized=True))

SORT BY [col("pickup_hour")]
  AGGREGATE[maintain_order: false]
    [len().alias("trip_count"), col("trip_distance").mean().alias("avg_distance"), col("fare_amount").mean().alias("avg_fare"), col("tip_ratio").mean().alias("avg_tip_ratio")] BY [col("pickup_hour")]
    FROM
    simple π 4/4 ["pickup_hour", "trip_distance", ... 2 other columns]
       WITH_COLUMNS:
       [col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"), (col("tip_amount") / col("fare_amount")).alias("tip_ratio")] 
        Parquet SCAN [data/yellow_tripdata_2024-01.parquet]
        PROJECT 4/19 COLUMNS
        SELECTION: (col("fare_amount") > 0.0) & (col("trip_distance") > 0.0)
        ESTIMATED ROWS: 2964624


## 34. Execute with `collect`

In [166]:
lazy_result = lazy_query.collect()
lazy_result

pickup_hour,trip_count,avg_distance,avg_fare,avg_tip_ratio
i8,u32,f64,f64,f64
0,75241,3.855185,19.697871,0.200051
1,50481,3.267594,17.735544,0.199491
2,34961,3.039499,16.629976,0.202416
3,22942,3.516815,18.536653,0.367911
4,15276,4.870405,23.451219,0.169535
…,…,…,…,…
19,178791,3.159724,17.640041,0.228588
20,155544,3.362001,18.05321,0.22171
21,155895,3.458705,18.314423,0.218805


## 35. Streaming execution

In [169]:
streaming_result = lazy_query.collect(engine="streaming")
streaming_result

pickup_hour,trip_count,avg_distance,avg_fare,avg_tip_ratio
i8,u32,f64,f64,f64
0,75241,3.855185,19.697871,0.200051
1,50481,3.267594,17.735544,0.199491
2,34961,3.039499,16.629976,0.202416
3,22942,3.516815,18.536653,0.367911
4,15276,4.870405,23.451219,0.169535
…,…,…,…,…
19,178791,3.159724,17.640041,0.228588
20,155544,3.362001,18.05321,0.22171
21,155895,3.458705,18.314423,0.218805


# Part X — Advanced Expressions

## 36. Expression reuse

In [173]:
valid_trip = (
    (pl.col("trip_distance") > 0) &
    (pl.col("fare_amount") > 0) &
    (pl.col("total_amount") > 0)
)

df.filter(valid_trip).head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


## 37. Horizontal calculations

In [176]:
df.select(
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    pl.sum_horizontal(
        "fare_amount",
        "tip_amount",
        "tolls_amount",
    ).alias("selected_charge_sum"),
).head()

fare_amount,tip_amount,tolls_amount,selected_charge_sum
f64,f64,f64,f64
17.7,0.0,0.0,17.7
10.0,3.75,0.0,13.75
23.3,3.0,0.0,26.3
10.0,2.0,0.0,12.0
7.9,3.2,0.0,11.1


## 38. Column name patterns

In [179]:
df.select(
    pl.col("^.*amount.*$")
).head()

fare_amount,tip_amount,tolls_amount,total_amount
f64,f64,f64,f64
17.7,0.0,0.0,22.7
10.0,3.75,0.0,18.75
23.3,3.0,0.0,31.3
10.0,2.0,0.0,17.0
7.9,3.2,0.0,16.1


# Part XI — SQL with Polars

## 39. SQLContext

Polars also provides a SQL interface, useful when translating familiar SQL logic into Polars workflows.

In [183]:
ctx = pl.SQLContext(trips=df)

sql_result = ctx.execute(
    '''
    SELECT
        payment_type,
        COUNT(*) AS trip_count,
        AVG(trip_distance) AS avg_trip_distance,
        AVG(total_amount) AS avg_total_amount
    FROM trips
    WHERE trip_distance > 0
    GROUP BY payment_type
    ORDER BY trip_count DESC
    '''
).collect()

sql_result

payment_type,trip_count,avg_trip_distance,avg_total_amount
i64,i64,f64,f64
1,2298442,3.293914,28.074859
2,430608,3.324087,22.96276
0,117337,13.945369,26.096042
4,42835,3.418642,1.684406
3,15031,2.815129,9.005361


# Part XIII — Reading and Writing Files

## 40. Write Parquet

In [190]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

payment_summary.write_parquet(
    OUTPUT_DIR / "payment_summary.parquet"
)

## 41. Write CSV

In [193]:
payment_summary.write_csv(
    OUTPUT_DIR / "payment_summary.csv"
)

## 42. Sink a lazy result directly to Parquet

In [196]:
# Uncomment when you want to write the lazy result.
# lazy_query.sink_parquet(OUTPUT_DIR / "hourly_summary.parquet")

# Part XIV — Practical Data Quality Pipeline

## 43. Build a cleaned taxi dataset

This combines several common Polars patterns into one reusable transformation pipeline.

In [200]:
clean_trips = (
    pl.scan_parquet(TRIP_FILE)
    .filter(
        (pl.col("trip_distance") > 0) &
        (pl.col("fare_amount") > 0) &
        (pl.col("total_amount") > 0) &
        (pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"))
    )
    .with_columns(
        (
            pl.col("tpep_dropoff_datetime") -
            pl.col("tpep_pickup_datetime")
        ).dt.total_minutes().alias("trip_minutes"),
        pl.col("tpep_pickup_datetime").dt.date().alias("pickup_date"),
        pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
        pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_weekday"),
        pl.when(pl.col("fare_amount") > 0)
          .then(pl.col("tip_amount") / pl.col("fare_amount"))
          .otherwise(None)
          .alias("tip_ratio"),
    )
    .filter(
        (pl.col("trip_minutes") > 0) &
        (pl.col("trip_minutes") <= 240)
    )
)

clean_trips.select(
    pl.len().alias("clean_row_count")
).collect()

clean_row_count
u32
2859124


# Part XV — Performance-Minded Polars Patterns

## Prefer expressions over Python loops

Use native expressions:

```python
df.with_columns(
    (pl.col("fare_amount") * 1.05).alias("adjusted_fare")
)
```

## Prefer lazy scanning for large data

Use:

```python
pl.scan_parquet(...)
```

instead of immediately loading everything with `pl.read_parquet(...)` when building a transformation pipeline.

## Select only what you need

Projection pushdown lets Polars avoid reading unnecessary columns.

## Filter early

Predicate pushdown can let Polars avoid processing irrelevant rows.

## Avoid Python UDFs when a native expression exists

Native Polars expressions are generally faster and easier for the optimizer to reason about.

## Prefer Parquet for analytical pipelines

Parquet is columnar, compressed, and supports efficient column pruning.

# Part XVI — Polars vs pandas Mental Model

| Task | pandas | Polars |
|---|---|---|
| Read CSV | `pd.read_csv()` | `pl.read_csv()` |
| Read Parquet | `pd.read_parquet()` | `pl.read_parquet()` |
| Lazy Parquet | — | `pl.scan_parquet()` |
| Select | `df[["a","b"]]` | `df.select("a","b")` |
| Filter | `df[df["a"] > 0]` | `df.filter(pl.col("a") > 0)` |
| New column | `df["c"] = ...` | `df.with_columns(...alias("c"))` |
| Group | `df.groupby()` | `df.group_by()` |
| Sort | `df.sort_values()` | `df.sort()` |
| Join | `df.merge()` | `df.join()` |
| Missing | `.isna()` | `.is_null()` |
| Unique | `.unique()` | `.unique()` |
| Execution | mostly eager | eager + lazy |

The biggest conceptual shift is to think in **expressions** rather than manipulating one Series at a time.

# Part XVII — Key Polars Syntax Bank

```python
# DataFrame
pl.DataFrame(...)

# Eager reads
pl.read_csv(...)
pl.read_parquet(...)

# Lazy reads
pl.scan_csv(...)
pl.scan_parquet(...)

# Expressions
pl.col("column")
pl.all()
pl.lit(value)

# Selection and transformation
df.select(...)
df.with_columns(...)
df.filter(...)
df.sort(...)

# Aggregation
df.group_by(...).agg(...)

# Combining
df.join(...)
pl.concat([...])

# Conditional
pl.when(...).then(...).otherwise(...)

# Missing
pl.col("x").is_null()
pl.col("x").fill_null(...)

# Datetime
pl.col("date").dt.year()
pl.col("date").dt.month()
pl.col("date").dt.weekday()
pl.col("date").dt.hour()

# Strings
pl.col("text").str.contains(...)
pl.col("text").str.to_lowercase()

# Window
pl.col("x").mean().over("group")

# Lazy execution
query.collect()
query.explain()

# Output
df.write_csv(...)
df.write_parquet(...)
```